# NER — PhoBERT Token Classification

Same pipeline as the ViSoBERT NER notebook (`ner_visobertv2.ipynb`), but with **`vinai/phobert-base`**.

- Data: per-file 70/15/15 split + augmented weak-class samples
- Imbalance: no-entity product_ner downsampled to ≤35%
- Loss: Focal Loss (γ=2) + class weights

**Why this notebook differs in tokenisation:** PhoBERT's tokenizer is a *slow* BPE tokenizer and does **not** support `return_offsets_mapping` or `word_ids()` (which the ViSoBERT notebook relied on). So tokenisation+alignment **and** inference are done manually at the **word level**: split on whitespace, derive each word's BIO tag from the entity char-spans, BPE-encode each word, and give the label to its first sub-token.

**Note:** word segmentation (underthesea/VnCoreNLP) is intentionally *not* applied here, to keep a fair ViSoBERT-vs-PhoBERT comparison. Adding it is a possible later enhancement.

In [1]:
!pip install transformers torch seqeval scikit-learn hf_xet --quiet

In [ ]:
# Running locally — uncomment if you switch to a Colab kernel and need Drive data.
#from google.colab import drive
#drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import json, os, re, random
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from seqeval.metrics import classification_report, f1_score as seqeval_f1
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME   = 'vinai/phobert-base'        # <-- PhoBERT (was uitnlp/visobert)
MAX_LENGTH   = 128
BATCH_SIZE   = 16
LR           = 2e-5
EPOCHS       = 100
PATIENCE     = 10
WARMUP_RATIO = 0.1
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

os.makedirs('results', exist_ok=True)

Device: cpu


In [4]:
# 13 entity types -> 27 classes (O + B/I x 13)
ENTITY_TYPES = [
    'NAME', 'PHONE', 'ADDRESS', 'CITY',
    'PRODUCT_NAME', 'SHIP_DATE', 'SHIP_TIME', 'TYPE',
    'COMPLEXITY', 'QUANTITY', 'PRODUCT_COLOR',
    'MAX_BUDGET', 'MIN_BUDGET',
]
LABELS   = ['O'] + [f'{p}-{e}' for e in ENTITY_TYPES for p in ('B', 'I')]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
print(f'Total labels: {NUM_LABELS}')
print(LABELS)

Total labels: 27
['O', 'B-NAME', 'I-NAME', 'B-PHONE', 'I-PHONE', 'B-ADDRESS', 'I-ADDRESS', 'B-CITY', 'I-CITY', 'B-PRODUCT_NAME', 'I-PRODUCT_NAME', 'B-SHIP_DATE', 'I-SHIP_DATE', 'B-SHIP_TIME', 'I-SHIP_TIME', 'B-TYPE', 'I-TYPE', 'B-COMPLEXITY', 'I-COMPLEXITY', 'B-QUANTITY', 'I-QUANTITY', 'B-PRODUCT_COLOR', 'I-PRODUCT_COLOR', 'B-MAX_BUDGET', 'I-MAX_BUDGET', 'B-MIN_BUDGET', 'I-MIN_BUDGET']


## 1. Data Loading — per-file split (Option B)

Split each source file independently at 70/15/15, then merge. Ensures every entity type appears in all splits.

In [ ]:
DATA_DIR = Path('../ner_data')

def split_file(records, train_r=0.70, val_r=0.15, seed=SEED):
    idx = list(range(len(records)))
    random.Random(seed).shuffle(idx)
    n_tr = int(len(records) * train_r)
    n_va = int(len(records) * val_r)
    return (
        [records[i] for i in idx[:n_tr]],
        [records[i] for i in idx[n_tr:n_tr + n_va]],
        [records[i] for i in idx[n_tr + n_va:]],
    )

def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def downsample_no_entity(records, max_ratio=0.35, seed=SEED):
    # Keep all entity samples; downsample no-entity to at most max_ratio of total.
    with_ent = [r for r in records if r['entities']]
    no_ent   = [r for r in records if not r['entities']]
    max_keep = int(len(with_ent) * max_ratio / (1 - max_ratio))
    kept     = random.Random(seed).sample(no_ent, min(len(no_ent), max_keep))
    return with_ent + kept

product_records = load_json(DATA_DIR / 'product_ner.json')
info_records    = load_json(DATA_DIR / 'info_ner.json')
budget_records  = load_json(DATA_DIR / 'budget_ner.json')

for aug_path, target in [
    (DATA_DIR / 'augmented_product_ner.json', 'product'),
    (DATA_DIR / 'augmented_info_ner.json',    'info'),
    (DATA_DIR / 'augmented_budget_ner.json',  'budget'),
]:
    if aug_path.exists():
        aug = load_json(aug_path)
        if target == 'product':
            product_records += aug
        elif target == 'info':
            info_records += aug
        elif target == 'budget':
            budget_records += aug
        print(f'Loaded {len(aug)} augmented {target}_ner samples')

before = len(product_records)
product_records = downsample_no_entity(product_records)
no_ent_pct = sum(1 for r in product_records if not r['entities']) / len(product_records) * 100
print(f'product_ner: {before} -> {len(product_records)} after downsampling ({no_ent_pct:.0f}% no-entity)')

all_train, all_val, all_test = [], [], []
for fname, records in [('info_ner', info_records),
                        ('product_ner', product_records),
                        ('budget_ner', budget_records)]:
    tr, va, te = split_file(records)
    all_train += tr
    all_val   += va
    all_test  += te
    print(f'{fname}: {len(records):>4} total  ->  {len(tr):>4} train / {len(va):>3} val / {len(te):>3} test')

random.shuffle(all_train)
print(f'\nCombined: {len(all_train)} train / {len(all_val)} val / {len(all_test)} test')

Loaded 415 augmented product_ner samples
Loaded 50 augmented info_ner samples
Loaded 80 augmented budget_ner samples
product_ner: 2243 -> 1978 after downsampling (35% no-entity)
info_ner:  243 total  ->   170 train /  36 val /  37 test
product_ner: 1978 total  ->  1384 train / 296 val / 298 test
budget_ner:  204 total  ->   142 train /  30 val /  32 test

Combined: 1696 train / 362 val / 367 test


## 2. Tokenisation + BIO Alignment (word-level, PhoBERT-compatible)

PhoBERT's tokenizer has no `offset_mapping`/`word_ids`, so we align manually:
split the text on whitespace, find each word's BIO tag from the entity char-spans,
BPE-encode each word, and label only its **first** sub-token (`-100` for the rest).

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_and_align(record):
    # Word-level alignment (PhoBERT has no offset_mapping/word_ids).
    text     = record['text']
    entities = sorted(record.get('entities', []), key=lambda e: e['start'])

    body_ids, body_labels = [], []
    for m in re.finditer(r'\S+', text):
        word, ws, we = m.group(), m.start(), m.end()
        tag = 'O'
        for ent in entities:
            if ws >= ent['start'] and we <= ent['end']:   # word fully inside entity span
                tag = ('B' if ws == ent['start'] else 'I') + '-' + ent['label']
                break
        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if not sub_ids:
            continue
        body_ids.append(sub_ids[0])
        body_labels.append(label2id.get(tag, label2id['O']))       # first sub-token -> label
        body_ids.extend(sub_ids[1:])
        body_labels.extend([-100] * (len(sub_ids) - 1))            # continuations -> ignore

    body_ids    = body_ids[:MAX_LENGTH - 2]      # leave room for <s> / </s>
    body_labels = body_labels[:MAX_LENGTH - 2]
    input_ids   = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]
    labels      = [-100] + body_labels + [-100]
    return {
        'input_ids'     : input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels'        : labels,
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [7]:
# Smoke test — show each word, its first sub-token, and the BIO label.
sample_rec = next(r for r in all_train if r['entities'])
text     = sample_rec['text']
entities = sorted(sample_rec['entities'], key=lambda e: e['start'])
print(f'Text: {text!r}\n')
print(f'{"Word":<16}{"1st sub-token":<18}Label')
for m in re.finditer(r'\S+', text):
    word, ws, we = m.group(), m.start(), m.end()
    tag = 'O'
    for ent in entities:
        if ws >= ent['start'] and we <= ent['end']:
            tag = ('B' if ws == ent['start'] else 'I') + '-' + ent['label']
            break
    sub = tokenizer.convert_ids_to_tokens(tokenizer.encode(word, add_special_tokens=False))
    print(f'  {word:<16}{(sub[0] if sub else ""):<18}{tag}')

Text: 'Còn bộ Porsche 911 màu xanh lá cây ko shop?'

Word            1st sub-token     Label
  Còn             Còn               O
  bộ              bộ                O
  Porsche         Porsche           B-PRODUCT_NAME
  911             911               I-PRODUCT_NAME
  màu             màu               O
  xanh            xanh              B-PRODUCT_COLOR
  lá              lá                I-PRODUCT_COLOR
  cây             cây               I-PRODUCT_COLOR
  ko              ko                O
  shop?           sho@@             O


## 3. Dataset + DataLoader

In [8]:
def maybe_lowercase(record, p=0.3):
    # .lower() preserves char count for Vietnamese, so char offsets stay valid.
    if random.random() < p:
        return {**record, 'text': record['text'].lower()}
    return record

class NERDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment   # True only for train split
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        if self.augment:
            rec = maybe_lowercase(rec)   # fresh roll each epoch
        return tokenize_and_align(rec)

def collate_fn(batch):
    max_len = max(len(s['input_ids']) for s in batch)
    input_ids, masks, labels = [], [], []
    for s in batch:
        pad = max_len - len(s['input_ids'])
        input_ids.append(s['input_ids'] + [tokenizer.pad_token_id] * pad)
        masks.append(s['attention_mask'] + [0] * pad)
        labels.append(s['labels'] + [-100] * pad)
    return {
        'input_ids'     : torch.tensor(input_ids),
        'attention_mask': torch.tensor(masks),
        'labels'        : torch.tensor(labels),
    }

train_ds = NERDataset(all_train, augment=True)
val_ds   = NERDataset(all_val,   augment=False)
test_ds  = NERDataset(all_test,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f'Batches - train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')
print('Case augmentation: ON (p=0.30) for train | OFF for val/test')

Batches - train: 106, val: 23, test: 23
Case augmentation: ON (p=0.30) for train | OFF for val/test


## 4. Class Weights

`O` dominates; weighted loss downweights it so the model is penalised more for missing real entities.

In [9]:
label_counts = Counter()
for rec in all_train:
    for lab in tokenize_and_align(rec)['labels']:
        if lab != -100:
            label_counts[lab] += 1

total   = sum(label_counts.values())
weights = torch.ones(NUM_LABELS)
for lab_id, cnt in label_counts.items():
    weights[lab_id] = total / (NUM_LABELS * cnt)

O_IDX = label2id['O']
weights[O_IDX] = min(weights[O_IDX].item(), 0.5)
weights = weights.to(DEVICE)

print('O tag weight :',  round(weights[O_IDX].item(), 4))
print('Top-5 highest weights:')
for i in sorted(range(NUM_LABELS), key=lambda i: -weights[i].item())[:5]:
    print(f'  {LABELS[i]:<25} {weights[i].item():.3f}')

O tag weight : 0.0497
Top-5 highest weights:
  I-PHONE                   203.383
  I-CITY                    61.015
  B-CITY                    26.528
  I-MIN_BUDGET              21.040
  B-PHONE                   15.645


## 5. Model Setup

In [10]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    # Focal Loss — down-weights easy/frequent examples, focuses on hard/rare ones (gamma=2).
    def __init__(self, weight=None, gamma=2.0, ignore_index=-100):
        super().__init__()
        self.weight       = weight
        self.gamma        = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               ignore_index=self.ignore_index, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        mask = targets != self.ignore_index
        return loss[mask].mean()


model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion    = FocalLoss(weight=weights, gamma=2.0, ignore_index=-100)

print(f'Parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'Train steps: {total_steps}  |  Warmup: {warmup_steps}')
print(f'Loss       : FocalLoss (gamma=2.0) + class weights')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters : 134,428,443
Train steps: 10600  |  Warmup: 1060
Loss       : FocalLoss (gamma=2.0) + class weights


## 6. Training

In [11]:
def predict_tags(loader):
    # Return (all_preds, all_trues) as lists of tag-string sequences (first sub-token only).
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch['input_ids'].to(DEVICE),
                attention_mask = batch['attention_mask'].to(DEVICE),
            ).logits
            preds  = logits.argmax(-1).cpu()
            labels = batch['labels']
            for pred_row, true_row in zip(preds, labels):
                p_seq, t_seq = [], []
                for p, t in zip(pred_row.tolist(), true_row.tolist()):
                    if t != -100:
                        p_seq.append(id2label[p])
                        t_seq.append(id2label[t])
                all_preds.append(p_seq)
                all_trues.append(t_seq)
    return all_preds, all_trues

def evaluate_f1(loader):
    preds, trues = predict_tags(loader)
    return seqeval_f1(trues, preds, average='micro', zero_division=0)

In [12]:
train_losses, val_f1s = [], []
best_val_f1, best_epoch, patience_ctr = 0.0, 0, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(
            input_ids      = batch['input_ids'].to(DEVICE),
            attention_mask = batch['attention_mask'].to(DEVICE),
        ).logits
        loss = criterion(logits.view(-1, NUM_LABELS), batch['labels'].to(DEVICE).view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    val_f1   = evaluate_f1(val_loader)
    train_losses.append(avg_loss)
    val_f1s.append(val_f1)

    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1, best_epoch, patience_ctr = val_f1, epoch, 0
        torch.save(model.state_dict(), 'results/ner_phobert_best_model.pt')
    else:
        patience_ctr += 1

    flag = ' *' if improved else ''
    print(f'Epoch {epoch:>2}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Micro-F1: {val_f1:.4f}{flag}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (best: epoch {best_epoch})')
        break

print(f'\nBest Val Micro-F1: {best_val_f1:.4f} at epoch {best_epoch}')

Epoch  1/100 | Loss: 3.1160 | Val Micro-F1: 0.0072 *
Epoch  2/100 | Loss: 3.0128 | Val Micro-F1: 0.0372 *
Epoch  3/100 | Loss: 2.7226 | Val Micro-F1: 0.0872 *


: 

## 7. Training Curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2      = ax1.twinx()
epochs_x = list(range(1, len(train_losses) + 1))

ax1.plot(epochs_x, train_losses, 'b-o', markersize=4, label='Train Loss')
ax2.plot(epochs_x, val_f1s,     'r-s', markersize=4, label='Val Micro-F1')
ax1.axvline(best_epoch, color='gray', linestyle='--', alpha=0.5)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss',         color='b')
ax2.set_ylabel('Val Micro-F1', color='r')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.title(f'PhoBERT NER — Training Curves (best epoch {best_epoch})')
plt.tight_layout()
plt.savefig('results/ner_phobert_training_curves.png', dpi=150)
plt.show()

## 8. Test Evaluation (entity-level F1 via seqeval)

In [ ]:
model.load_state_dict(torch.load('results/ner_phobert_best_model.pt', map_location=DEVICE))
test_preds, test_trues = predict_tags(test_loader)

report = classification_report(test_trues, test_preds, digits=4, zero_division=0)
print(report)

with open('results/ner_phobert_test_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

## 9. Save Model

In [ ]:
os.makedirs('results/ner_phobert_model', exist_ok=True)
model.save_pretrained('results/ner_phobert_model')
tokenizer.save_pretrained('results/ner_phobert_model')
print('Saved to results/ner_phobert_model/')

## 10. Inference Demo

PhoBERT inference also aligns at the word level: encode word by word, take each word's
**first** sub-token prediction, then BIO-chunk over words. (Reading continuation
sub-tokens would fragment entities, since they are never trained.)

In [ ]:
def predict_ner(text: str) -> list:
    model.eval()
    words = [(m.group(), m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    body_ids, first_word = [], []      # first_word[k] = word index if first sub-token else -1
    for wi, (w, ws, we) in enumerate(words):
        sub_ids = tokenizer.encode(w, add_special_tokens=False)
        if not sub_ids:
            continue
        body_ids.append(sub_ids[0]); first_word.append(wi)
        for sid in sub_ids[1:]:
            body_ids.append(sid); first_word.append(-1)
    body_ids   = body_ids[:MAX_LENGTH - 2]
    first_word = first_word[:MAX_LENGTH - 2]
    input_ids  = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]

    with torch.no_grad():
        logits = model(input_ids=torch.tensor([input_ids]).to(DEVICE)).logits[0]
    preds = logits.argmax(-1).tolist()[1:-1]     # drop <s> / </s>

    word_tags = {}
    for pred, wi in zip(preds, first_word):
        if wi >= 0:
            word_tags[wi] = id2label[pred]

    entities, cur = [], None
    for wi, (w, ws, we) in enumerate(words):
        tag = word_tags.get(wi, 'O')
        if tag == 'O':
            if cur:
                entities.append(cur); cur = None
        elif tag.startswith('B-'):
            if cur:
                entities.append(cur)
            cur = {'label': tag[2:], 'start': ws, 'end': we}
        else:
            etype = tag[2:]
            if cur and cur['label'] == etype:
                cur['end'] = we
            else:
                if cur:
                    entities.append(cur)
                cur = {'label': etype, 'start': ws, 'end': we}
    if cur:
        entities.append(cur)

    for ent in entities:
        ent['text'] = text[ent['start']:ent['end']]
    return entities

In [ ]:
test_cases = [
    'Nguyên Thảo\n0912495077\nK45/10B Dũng Sĩ Thanh Khê',
    'bạn có bộ nào tầm 200k không ạ',
    'mình muốn đặt bộ lego city, ship ngày mai nhé',
    'tên: Phạm Thành\nsdt: 0949913458\ndchi: 46 Sơn Hải Đồ Sơn Hải Phòng',
    'em ở hn ạ',
    'cho em đặt 1 bộ mẹc f1 về Cát Linh với ạ'
]
for text in test_cases:
    ents = predict_ner(text)
    print(f'Input : {text!r}')
    if ents:
        for e in ents:
            print(f'  {e["label"]:<15} {e["text"]!r}')
    else:
        print('  (no entities)')
    print()

In [ ]:
def extract_entities_from_text(text, model, tokenizer, id2label, device='cuda'):
    # Word-level aggregation -> {type: [values]} dict (deduplicated). New B- starts a new chunk.
    model.eval()
    words = [(m.group(), m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    body_ids, first_word = [], []
    for wi, (w, ws, we) in enumerate(words):
        sub_ids = tokenizer.encode(w, add_special_tokens=False)
        if not sub_ids:
            continue
        body_ids.append(sub_ids[0]); first_word.append(wi)
        for sid in sub_ids[1:]:
            body_ids.append(sid); first_word.append(-1)
    body_ids   = body_ids[:MAX_LENGTH - 2]
    first_word = first_word[:MAX_LENGTH - 2]
    input_ids  = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]

    with torch.no_grad():
        logits = model(input_ids=torch.tensor([input_ids]).to(device)).logits[0]
    preds = logits.argmax(-1).tolist()[1:-1]

    word_tags = {}
    for pred, wi in zip(preds, first_word):
        if wi >= 0:
            word_tags[wi] = id2label[pred]

    chunks, cur = [], None
    for wi, (w, ws, we) in enumerate(words):
        tag = word_tags.get(wi, 'O')
        if tag == 'O':
            if cur:
                chunks.append(cur); cur = None
            continue
        prefix, etype = tag.split('-', 1)
        if prefix == 'B' or cur is None or cur['type'] != etype:
            if cur:
                chunks.append(cur)
            cur = {'type': etype, 'start': ws, 'end': we}
        else:
            cur['end'] = we
    if cur:
        chunks.append(cur)

    final_results = {}
    for c in chunks:
        value = text[c['start']:c['end']].strip().lstrip(':').strip()
        if not value:
            continue
        final_results.setdefault(c['type'], [])
        if value not in final_results[c['type']]:
            final_results[c['type']].append(value)
    return final_results

In [ ]:
test_input = 'có bộ nào cho bé nam tầm 2-300k không shop'
entities = extract_entities_from_text(test_input, model, tokenizer, id2label, device=DEVICE)
for entity_type, values in entities.items():
    for val in values:
        print(f"{entity_type:<15} '{val}'")

In [ ]:
# Reload from the saved local weights (after a kernel restart)
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('results/ner_phobert_model', use_fast=False)
model     = AutoModelForTokenClassification.from_pretrained('results/ner_phobert_model')
model.to(DEVICE)
model.eval()
id2label = model.config.id2label